In [0]:
gold_tables = [
    row.tableName
    for row in spark.sql(
        "SHOW TABLES IN globalretail.gold"
    ).collect()
]

for table in gold_tables:
    df = spark.table(f"globalretail.gold.{table}")
    print(f"{table}: {df.count()} rows")
    df.printSchema()

In [0]:
%sql
SHOW TABLES IN globalretail_snowflake_conn_catalog.marts;

In [0]:
snowflake_marts = (
    "globalretail_snowflake_conn_catalog.marts"
)
for row in spark.sql(
    f"SHOW TABLES IN {snowflake_marts}"
).collect():
    print(row.tableName)


In [0]:
%pip install snowflake-connector-python

%pip install snowflake-sqlalchemy

In [0]:
%restart_python

In [0]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

# --- Replace these with your Snowflake credentials ---
SF_ACCOUNT = "AJVJTMY-YX45013"         # from UC connection host
SF_USER = "MOBIOUSUPRETI"        # your Snowflake username
SF_PASSWORD = "Prathamesh.00."  # your Snowflake password
SF_ROLE = "ACCOUNTADMIN"        # e.g. ACCOUNTADMIN
SF_WAREHOUSE = "COMPUTE_WH"  # e.g. COMPUTE_WH
SF_DATABASE = "GLOBALRETAIL_DB"      # Snowflake database
SF_SCHEMA = "MARTS"                  # target schema for marts

# Connect to Snowflake
sf_conn = snowflake.connector.connect(
    account=SF_ACCOUNT,
    user=SF_USER,
    password=SF_PASSWORD,
    role=SF_ROLE,
    warehouse=SF_WAREHOUSE,
    database=SF_DATABASE,
    schema=SF_SCHEMA,
)

# Test the connection
sf_cursor = sf_conn.cursor()
sf_cursor.execute("SELECT CURRENT_VERSION()")
print("Connected to Snowflake! Version:", sf_cursor.fetchone()[0])
sf_cursor.close()

In [0]:
from pyspark.sql.types import (
    StringType, IntegerType, LongType, DoubleType,
    DecimalType, DateType, TimestampType, BooleanType,
)

# Map Spark column types to Snowflake types
def spark_to_snowflake_type(dt):
    if isinstance(dt, StringType): return "VARCHAR"
    if isinstance(dt, IntegerType): return "INTEGER"
    if isinstance(dt, LongType): return "BIGINT"
    if isinstance(dt, DoubleType): return "DOUBLE"
    if isinstance(dt, DecimalType): return f"NUMBER({dt.precision},{dt.scale})"
    if isinstance(dt, DateType): return "DATE"
    if isinstance(dt, TimestampType): return "TIMESTAMP"
    if isinstance(dt, BooleanType): return "BOOLEAN"
    return "VARCHAR"

# Get list of gold tables from Databricks
gold_tables = [
    row.tableName
    for row in spark.sql("SHOW TABLES IN globalretail.gold").collect()
]

for table in gold_tables:
    df = spark.table(f"globalretail.gold.{table}")
    table_name = table.upper()

    # 1. Create table in Snowflake with matching column types
    col_defs = ", ".join(
        [f'"{f.name}" {spark_to_snowflake_type(f.dataType)}' for f in df.schema.fields]
    )
    sf_cursor = sf_conn.cursor()
    sf_cursor.execute(f"CREATE OR REPLACE TABLE {table_name} ({col_defs})")

    # 2. Collect data from Spark and insert into Snowflake
    rows = [tuple(r) for r in df.collect()]
    if rows:
        col_names = ", ".join([f'"{c}"' for c in df.columns])
        placeholders = ", ".join(["%s"] * len(df.columns))
        sf_cursor.executemany(
            f"INSERT INTO {table_name} ({col_names}) VALUES ({placeholders})",
            rows,
        )

    print(f"Loaded {table} ({len(rows)} rows) -> Snowflake {SF_DATABASE}.{SF_SCHEMA}.{table_name}")
    sf_cursor.close()